# Study 824 — Cochrane-Piazzesi Factor 🧮📉

**Does a single tent of forward rates forecast bond excess returns?**

Cochrane & Piazzesi (2005) regress each Treasury bond's one-year-ahead **excess return** on
the whole vector of **forward rates** and find the fitted values collapse onto *one*
tent-shaped factor `CP = γ'f` that forecasts excess returns of every maturity — an R² a plain
curve slope can't touch. We rebuild it from the coarse constant-maturity yields yfinance
exposes (`^IRX` 0.25y, `^FVX` 5y, `^TNX` 10y, `^TYX` 30y, 2002-01-02 → 2026-06-30) and
forecast the average 252-day excess return of the SHY/IEF/TLT bond ETFs.

*Numbers below are the frozen headline (`docs/results.md`, fingerprint `03a5e9844a31`);
the live cells run the fast synthetic control. Signal caveat: a coarse 4-forward proxy of CP's
Fama-Bliss 1..5y zeros — named on the Signal axis.*


## 1. The idea in one picture

The **expectations hypothesis** says a long yield is just the average expected future short rate — so *no* combination of today's forwards should predict a bond's future *excess* return. Cochrane & Piazzesi found otherwise: regress next year's excess return on all the forwards and one **tent-shaped** loading vector (down-short, up-middle, down-long) pops out that forecasts returns across every maturity. Fat premium ⇒ own duration; thin ⇒ step aside.

In [1]:
R = dict(r2=0.2261, cp_slope_t=2.618, load_yshort=-0.0587, load_f1=-2.1058, load_f2=4.9245, load_f3=-0.9744)
print('in-sample predictive R2 : %.3f' % R['r2'])
print('single-factor NW t      : %+.2f' % R['cp_slope_t'])
print('tent loadings  y_short=%+.2f  f_1=%+.2f  f_2=%+.2f  f_3=%+.2f  '
      '(peak on the 5->10y forward)'
      % (R['load_yshort'], R['load_f1'], R['load_f2'], R['load_f3']))

in-sample predictive R2 : 0.226
single-factor NW t      : +2.62
tent loadings  y_short=-0.06  f_1=-2.11  f_2=+4.92  f_3=-0.97  (peak on the 5->10y forward)


## 2. The catch — a fat R² on persistent yields is almost free

Yields are *near-unit-root persistent* and annual returns *overlap* 252-fold. Regress a persistent series on persistent regressors and you get a big R² **even with no true link** (Bauer-Hamilton 2018). Our live synthetic control makes this concrete: a **null** world (no forward→return link) vs a **planted** one. Watch the *R²* (the honest detector) and note the raw HAC *t* fires even on the null.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from cp_factor import data, strategy as st
null = st.synthetic_detect(data.synthetic_daily(edge=0.0, seed=824, n_days=3000))
plant = st.synthetic_detect(data.synthetic_daily(edge=0.05, seed=824, n_days=3000))
print('null world   : in-sample R2 = %.4f   (honest detector ~ 0)' % null['r2'])
print('planted world: in-sample R2 = %.4f   (lights up)' % plant['r2'])
print('null raw HAC t = %+.2f  <- size-distorted, fires even with no signal'
      % null['cp_slope_t'])

null world   : in-sample R2 = 0.0007   (honest detector ~ 0)
planted world: in-sample R2 = 0.6769   (lights up)
null raw HAC t = +1.30  <- size-distorted, fires even with no signal


## 3. The honest verdict — right shape, no robust edge

On the real tape the CP regression gives an in-sample **R² = 0.226** with the correct tent — the claim's fingerprint is visibly there. But it does **not** survive the honesty rails: a block placebo puts that R² at **p = 0.21** (pure persistence already delivers R² ≈ 0.18), the **out-of-sample R² is -0.27** (worse than a constant), the second era is insignificant (*t* = +1.72), and the headline HAC *t* = +2.62 sits inside the synthetic null's own band for that statistic (mean +2.08). A timed duration book earns a net Sharpe of ~0.02 vs 0.22 for just holding TLT. **Signal: Weak** (right tent, spurious-looking fit), **Tradability: Mirage**.